In [ ]:
import pandas as pd

tabela = pd.read_csv("DENGBR25.csv", low_memory=False)
# total dos registros de notificacao
total_notificacao = len(tabela)
print(f"Total de registros: {total_notificacao}")
# total dos registros de notificacao com criterio 1 ou 2 (confirmados por critério clínico ou laboratorial)
tabela = tabela[(tabela["CRITERIO"] == 1) | (tabela["CRITERIO"] == 2)]
total_confirmado = len(tabela)
print(f"Total de registros confirmados por critério clínico ou laboratorial: {total_confirmado}")



Total de registros: 1646533
Total de registros confirmados por critério clínico ou laboratorial: 1472044


<StringArray>
['32', '12', '52', '31', '51', '13', '41', '35', '42', '43', '50', '27', '26',
 '28', '29', '33', '21', '23', '25', '17', '24', '53', '15', '11', '14', 'am',
 '16', '22', 'go', 'mg', 'pa', 'pr', 'rj', 'rn', 'rs', 'sc']
Length: 36, dtype: str

In [12]:
# Dicionário de mapeamento: sigla -> código IBGE
sigla_para_codigo_ibge = {
    'AC': 12, 'AM': 13, 'AP': 16, 'PA': 15, 'RO': 11, 'RR': 14, 'TO': 17,  # Norte
    'MA': 21, 'PI': 22, 'CE': 23, 'RN': 24, 'PB': 25, 'PE': 26, 'AL': 27, 'SE': 28, 'BA': 29,  # Nordeste
    'MG': 31, 'ES': 32, 'RJ': 33, 'SP': 35,  # Sudeste
    'PR': 41, 'SC': 42, 'RS': 43,  # Sul
    'MS': 50, 'MT': 51, 'GO': 52, 'DF': 53  # Centro-Oeste
}

# Verificar valores únicos e identificar siglas em minúsculas
print("Valores únicos em SG_UF_NOT:")
print(tabela["SG_UF_NOT"].unique())
print(f"\nValores únicos: {sorted(tabela['SG_UF_NOT'].unique())}")

# Verificar quais são siglas em minúsculas
siglas_minusculas = [uf for uf in tabela["SG_UF_NOT"].unique() if isinstance(uf, str) and uf.islower()]
print(f"\nSiglas em minúsculas encontradas: {siglas_minusculas}")


Valores únicos em SG_UF_NOT:
<StringArray>
['32', '12', '52', '31', '51', '13', '41', '35', '42', '43', '50', '27', '26',
 '28', '29', '33', '21', '23', '25', '17', '24', '53', '15', '11', '14', 'am',
 '16', '22', 'go', 'mg', 'pa', 'pr', 'rj', 'rn', 'rs', 'sc']
Length: 36, dtype: str

Valores únicos: ['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26', '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52', '53', 'am', 'go', 'mg', 'pa', 'pr', 'rj', 'rn', 'rs', 'sc']

Siglas em minúsculas encontradas: ['am', 'go', 'mg', 'pa', 'pr', 'rj', 'rn', 'rs', 'sc']


In [14]:
# Corrigir siglas em minúsculas para códigos IBGE
def corrigir_e_normalizar_uf(valor):
    if isinstance(valor, str):
        valor_upper = valor.upper()
        # Se é uma sigla de 2 caracteres, converter para código IBGE
        if len(valor) == 2 and valor_upper in sigla_para_codigo_ibge:
            return str(sigla_para_codigo_ibge[valor_upper])
    return valor

# Aplicar correção
tabela["SG_UF_NOT"] = tabela["SG_UF_NOT"].apply(corrigir_e_normalizar_uf)

# Criar dicionário inverso: código IBGE (como string) -> sigla
codigo_para_sigla = {str(v): k for k, v in sigla_para_codigo_ibge.items()}

# Criar campo com sigla em maiúsculo
tabela["SIGLA_UF"] = tabela["SG_UF_NOT"].map(codigo_para_sigla)

# Verificar os dados após a correção
print("Valores únicos em SG_UF_NOT após correção:")
print(sorted(tabela["SG_UF_NOT"].unique()))
print(f"\nTotal de registros: {len(tabela)}")
print(f"\nAmostra dos dados corrigidos:")
print(tabela[["SG_UF_NOT", "SIGLA_UF"]].head(20))
print(f"\nVerificação de campos nulos em SIGLA_UF: {tabela['SIGLA_UF'].isna().sum()}")


Valores únicos em SG_UF_NOT após correção:
['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26', '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52', '53']

Total de registros: 1472044

Amostra dos dados corrigidos:
   SG_UF_NOT SIGLA_UF
0         32       ES
1         32       ES
2         32       ES
3         32       ES
4         32       ES
5         32       ES
6         32       ES
7         32       ES
8         32       ES
11        32       ES
12        32       ES
13        32       ES
14        32       ES
15        32       ES
16        32       ES
18        32       ES
19        32       ES
20        32       ES
21        32       ES
23        32       ES

Verificação de campos nulos em SIGLA_UF: 0


In [ ]:
# Verificar que as siglas que estavam em minúsculas foram corrigidas
print("Verificação das correções realizadas:")
print("\n--- Amazonas (AM) ---")
am_records = tabela[tabela["SIGLA_UF"] == "AM"]
print(f"Total de registros de AM: {len(am_records)}")
print(f"Código IBGE: {am_records['SG_UF_NOT'].unique()}")

print("\n--- Goiás (GO) ---")
go_records = tabela[tabela["SIGLA_UF"] == "GO"]
print(f"Total de registros de GO: {len(go_records)}")
print(f"Código IBGE: {go_records['SG_UF_NOT'].unique()}")

print("\n--- Resumo de todos os estados ---")
resumo = tabela.groupby("SIGLA_UF")["SG_UF_NOT"].agg(["count", "unique"])
print(resumo)





Verificação das correções realizadas:

--- Amazonas (AM) ---
Total de registros de AM: 4849
Código IBGE: <StringArray>
['13']
Length: 1, dtype: str

--- Goiás (GO) ---
Total de registros de GO: 101595
Código IBGE: <StringArray>
['52']
Length: 1, dtype: str

--- Resumo de todos os estados ---
           count unique
SIGLA_UF               
AC          7744   [12]
AL          7790   [27]
AM          4849   [13]
AP          2373   [16]
BA         17082   [29]
CE          4908   [23]
DF          4872   [53]
ES         32698   [32]
GO        101595   [52]
MA          3804   [21]
MG        120874   [31]
MS          9337   [50]
MT         30439   [51]
PA         15151   [15]
PB          7425   [25]
PE         14526   [26]
PI          7524   [22]
PR         93059   [41]
RJ         21257   [33]
RN          4054   [24]
RO          1848   [11]
RR           351   [14]
RS         52705   [43]
SC         18556   [42]
SE           691   [28]
SP        883952   [35]
TO          2580   [17]
